# 11.13 - RAG Evaluation

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

You cannot improve what you cannot measure. RAG evaluation separates retrieval quality (did we find the right docs?) from generation quality (did the answer use them well?) so you know which layer to fix.

## 2. Why Does This Matter?

Without ground-truth evaluation you guess. With recall@k / precision@k for retrieval and a keyword-overlap answer score for generation, you can iterate systematically.

## 3. Prerequisites

Units 11.1-11.12 (full RAG pipeline).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a self-contained eval set (queries, gold docs, gold answer keywords)
- Compute recall@k and precision@k for retrieval
- Compute a keyword-overlap answer score for generation
- Print a small results table and a precision bar chart

## 5. Mental Model

RAG evaluation is a two-part exam: (1) did the system find the right books? (2) did it use them to write a good answer? Grade each separately.

```text
Eval Set -> Run Pipeline -> Measure Retrieval -> Measure Generation -> Report
```


## 6. A Tiny Eval Set
Queries with (gold doc ids, gold answer keywords). All self-contained - no external harness needed.

In [1]:
EVAL_SET = [
    {"query": "how do I return something I bought",
     "gold_docs": ["d_returns"], "gold_keywords": ["30", "days", "return"]},
    {"query": "how long does standard shipping take",
     "gold_docs": ["d_shipping"], "gold_keywords": ["5", "7", "business", "days"]},
    {"query": "when does my refund arrive",
     "gold_docs": ["d_refunds"], "gold_keywords": ["5", "7", "refund"]},
]
print(f"{len(EVAL_SET)} evaluation queries")


3 evaluation queries


## 7. Retrieval Metrics (recall@k, precision@k)
We simulate a retriever that returns doc ids for a query (in a real pipeline this is your Chroma call), then score each query.

In [2]:
def fake_retriever(query):
    q = query.lower()
    mapping = {
        "return": ["d_returns"], "shipping": ["d_shipping"],
        "refund": ["d_refunds"], "warranty": ["d_warranty"],
    }
    hits = []
    for key, val in mapping.items():
        if key in q:
            hits.extend(val)
    return hits or ["d_warranty"]


def evaluate(eval_set, k):
    recall_total, prec_total = 0.0, 0.0
    rows = []
    for ex in eval_set:
        retrieved = fake_retriever(ex["query"])[:k]
        gold = set(ex["gold_docs"])
        rec_set = set(retrieved) & gold
        recall = len(rec_set) / len(gold)
        precision = len(rec_set) / k
        recall_total += recall
        prec_total += precision
        rows.append((ex["query"][:30], recall, precision, sorted(retrieved)))
    return rows, recall_total / len(eval_set), prec_total / len(eval_set)


## 8. Run Retrieval Eval + Table
We print a compact table of per-query recall/precision at k=3 plus the averages.

In [3]:
import pandas as pd
rows, avg_rec, avg_prec = evaluate(EVAL_SET, k=3)
df = pd.DataFrame(rows, columns=["query", "recall@3", "precision@3", "retrieved"])
print(df.to_string(index=False))
print(f"\navg recall@3 = {avg_rec:.2f}   avg precision@3 = {avg_prec:.2f}")


                         query  recall@3  precision@3    retrieved
how do I return something I bo       1.0     0.333333  [d_returns]
how long does standard shippin       1.0     0.333333 [d_shipping]
    when does my refund arrive       1.0     0.333333  [d_refunds]

avg recall@3 = 1.00   avg precision@3 = 0.33


## 9. Generation Score: Keyword Overlap
A cheap, self-contained generation metric: fraction of gold answer keywords present in the produced answer. (Production systems use faithfulness/relevance; this is the building block.)

In [4]:
def answer_keyword_score(answer, gold_keywords):
    if not gold_keywords:
        return 1.0
    a_lo = answer.lower()
    hit = sum(1 for kw in gold_keywords if kw.lower() in a_lo)
    return hit / len(gold_keywords)


answers = [
    ("return something I bought", "You can return within 30 days of purchase.", ["30", "days", "return"]),
    ("how long does standard shipping take", "Standard shipping takes 5 to 7 business days.", ["5", "7", "business", "days"]),
    ("when does my refund arrive", "Your refund may be lost.", ["5", "7", "refund"]),
]
print(f"{'query':32s} {'answer score':>12s}")
for q, a, kw in answers:
    print(f"{q[:32]:32s} {answer_keyword_score(a, kw):12.2f}")


query                            answer score
return something I bought                1.00
how long does standard shipping          1.00
when does my refund arrive               0.33


## 10. Precision Bar Chart
A quick matplotlib bar of per-query precision@3 makes regressions visible at a glance.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

recalls = [r[1] for r in rows]
plt.figure(figsize=(6, 3))
plt.bar(range(len(recalls)), recalls, color="#4C72B0")
plt.xticks(range(len(recalls)), [f"q{i}" for i in range(len(recalls))])
plt.ylim(0, 1)
plt.ylabel("recall@3")
plt.title("Retrieval recall@3 per query")
plt.tight_layout()
plt.show()


C:\Users\PC\AppData\Local\Temp\ipykernel_6700\54918288.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## Common Mistakes

- Only measuring answer quality, not retrieval quality.
- Not having a ground-truth evaluation set.
- Evaluating only on easy queries.
- Not tracking latency and cost.
- Using the same data for evaluation and testing.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Low recall | Missing relevant docs | Increase k, fix chunking/embedding |
| Low precision | Too many irrelevant results | Add reranking / filtering |
| Low faithfulness | LLM hallucinating despite context | Strengthen grounding |
| Slow evaluation | Huge test set | Sample the set |

## Best Practices

- Create a ground-truth eval set early (even 50 queries helps).
- Evaluate retrieval and generation separately.
- Track metrics over time, not just absolute values.
- Include latency and cost.
- Test adversarial queries (unanswerable, ambiguous).

## Hands-On Practice

1. **Basic:** Compute recall@3 for 5 queries.
2. **Guided:** Build a 20-query test set with ground truth.
3. **Independent:** Evaluate both retrieval and generation.
4. **Realistic:** Compare two retrieval strategies on the same set.
5. **Challenge:** Use RAGAS to compute faithfulness/relevance.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
